In [35]:
# Importing Libraries
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

# Loading Data
dataset = load_dataset('lukebarousse/data_jobs')
df = dataset['train'].to_pandas()

# Data Cleanup
df['job_posted_date'] = pd.to_datetime(df.job_posted_date)

## apply() - Calculate Project Salary Next Year

In [36]:
df['salary_year_avg']

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
          ..
785736   NaN
785737   NaN
785738   NaN
785739   NaN
785740   NaN
Name: salary_year_avg, Length: 785741, dtype: float64

In [37]:
df[pd.notna(df['salary_year_avg'])]['salary_year_avg']

28        109500.0
77        140000.0
92        120000.0
100       228222.0
109        89000.0
            ...   
785624    139216.0
785641    150000.0
785648    221875.0
785682    157500.0
785692    157500.0
Name: salary_year_avg, Length: 22003, dtype: float64

In [38]:
help(df.apply)

Help on method apply in module pandas.core.frame:

apply(func: 'AggFuncType', axis: 'Axis' = 0, raw: 'bool' = False, result_type: "Literal['expand', 'reduce', 'broadcast'] | None" = None, args=(), by_row: "Literal[False, 'compat']" = 'compat', engine: "Callable | None | Literal['python', 'numba']" = None, engine_kwargs: 'dict[str, bool] | None' = None, **kwargs) method of pandas.DataFrame instance
    Apply a function along an axis of the DataFrame.
    
    Objects passed to the function are Series objects whose index is
    either the DataFrame's index (``axis=0``) or the DataFrame's columns
    (``axis=1``). By default (``result_type=None``), the final return type
    is inferred from the return type of the applied function. Otherwise,
    it depends on the `result_type` argument. The return type of the applied
    function is inferred based on the first computed result obtained after
    applying the function to a Series object.
    
    Parameters
    ----------
    func : functio

In [39]:
# Let's assume there's a 3% inflation. Then, the project salary for the next year can be calculated as follows:

df_salary = df[pd.notna(df['salary_year_avg'])].copy()

def project_salary(salary):
    return salary * 1.03

df_salary['salary_year_inflated'] = df_salary['salary_year_avg'].apply(project_salary)
df_salary[['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


In [40]:
# Instead of defining a short function like 'project_salary()', we can use the lambda function

df_salary['salary_year_inflated'] = df_salary['salary_year_avg'].apply(lambda salary: salary * 1.03)

df_salary[['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


In [41]:
# The job_skills column should be changed from a string to a list

type(df['job_skills'][10])

str

In [42]:
# To convert a string datatype to a list datatype, use the ast module

import ast

type(ast.literal_eval(df['job_skills'][10]))

list

In [43]:
# To convert the entire column into a list, refer the following:

# df['job_skills'] = ast.literal_eval(df['job_skills'])

# The above doesn't work because the literal_eval() method requires a function

def clean_list(skills_list):
    if(pd.notna(skills_list)):
        return ast.literal_eval(skills_list)

df['job_skills'] = df['job_skills'].apply(clean_list)

df['job_skills'][0]

# df[pd.isna(df['job_skills'])]

In [48]:
type(df['job_skills'][10])

list

## Calculate project salary for next year

- Senior roles assume 5% inflation
- Other roles assume 3% inflation

In [ ]:
# To apply the inflation in condition of rows
# Now to apply the "apply()" function across the rows, use the axis parameter as against the func parameter used previously in case of columns
# Previously, axis = 0, meaning apply functionto every column of the data
# axis = 0: (or index) apply function to each column
# axis = 1: (or columns) apply function to each row

help(df.apply)

Help on method apply in module pandas.core.frame:

apply(func: 'AggFuncType', axis: 'Axis' = 0, raw: 'bool' = False, result_type: "Literal['expand', 'reduce', 'broadcast'] | None" = None, args=(), by_row: "Literal[False, 'compat']" = 'compat', engine: "Callable | None | Literal['python', 'numba']" = None, engine_kwargs: 'dict[str, bool] | None' = None, **kwargs) method of pandas.DataFrame instance
    Apply a function along an axis of the DataFrame.
    
    Objects passed to the function are Series objects whose index is
    either the DataFrame's index (``axis=0``) or the DataFrame's columns
    (``axis=1``). By default (``result_type=None``), the final return type
    is inferred from the return type of the applied function. Otherwise,
    it depends on the `result_type` argument. The return type of the applied
    function is inferred based on the first computed result obtained after
    applying the function to a Series object.
    
    Parameters
    ----------
    func : functio

In [51]:
def projected_salary(row):
    if 'Senior' in row['job_title_short']:
        return 1.05 * row['salary_year_avg']
    else:
        return 1.03 * row['salary_year_avg']

df_salary['salary_year_inflated'] = df_salary.apply(projected_salary, axis = 1)

df_salary[['job_title_short', 'salary_year_avg', 'salary_year_inflated']]

,job_title_short,salary_year_avg,salary_year_inflated
28,Data Scientist,109500.0,112785.00
77,Data Engineer,140000.0,144200.00
92,Data Engineer,120000.0,123600.00
100,Data Scientist,228222.0,235068.66
109,Data Analyst,89000.0,91670.00
...,...,...,...
785624,Data Engineer,139216.0,143392.48
785641,Data Engineer,150000.0,154500.00
785648,Data Scientist,221875.0,228531.25
785682,Data Scientist,157500.0,162225.00
